In [11]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

In [12]:
X = np.load(
    "../data/processed/AAPL_X.npy"
)

y = np.load(
    "../data/processed/AAPL_y.npy"
)

window_dates = pd.read_csv(
    "../data/processed/AAPL_window_dates.csv",
    parse_dates=["Date"]
)

dates = window_dates["Date"]

print("=" * 60)
print("LOADED SUPERVISED DATASET")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("dates shape:", dates.shape)

print(
    "\nDate range:",
    dates.min(),
    "→",
    dates.max()
)

LOADED SUPERVISED DATASET
X shape: (208, 22, 9)
y shape: (208,)
dates shape: (208,)

Date range: 2025-10-14 00:00:00 → 2026-08-12 00:00:00


In [13]:
assert X.shape[0] == len(y)
assert X.shape[0] == len(dates)

assert X.shape[1] == 22
assert X.shape[2] == 9

assert dates.is_monotonic_increasing
assert dates.is_unique

assert np.isfinite(X).all()
assert np.isfinite(y).all()

print("Loaded dataset validation: PASSED")

Loaded dataset validation: PASSED


In [14]:
n_samples = len(X)

train_end = int(
    n_samples * 0.60
)

validation_end = int(
    n_samples * 0.80
)

print(
    "Total samples:",
    n_samples
)

print(
    "Train samples:",
    train_end
)

print(
    "Validation samples:",
    validation_end - train_end
)

print(
    "Test samples:",
    n_samples - validation_end
)

Total samples: 208
Train samples: 124
Validation samples: 42
Test samples: 42


In [15]:
X_train = X[
    :train_end
]

y_train = y[
    :train_end
]

dates_train = dates[
    :train_end
]


X_val = X[
    train_end:validation_end
]

y_val = y[
    train_end:validation_end
]

dates_val = dates[
    train_end:validation_end
]


X_test = X[
    validation_end:
]

y_test = y[
    validation_end:
]

dates_test = dates[
    validation_end:
]

In [16]:
print("=" * 60)
print("CHRONOLOGICAL SPLIT")
print("=" * 60)

print(
    "TRAIN:",
    X_train.shape,
    f"| {dates_train.iloc[0].date()} → {dates_train.iloc[-1].date()}"
)

print(
    "VALIDATION:",
    X_val.shape,
    f"| {dates_val.iloc[0].date()} → {dates_val.iloc[-1].date()}"
)

print(
    "TEST:",
    X_test.shape,
    f"| {dates_test.iloc[0].date()} → {dates_test.iloc[-1].date()}"
)

CHRONOLOGICAL SPLIT
TRAIN: (124, 22, 9) | 2025-10-14 → 2026-04-13
VALIDATION: (42, 22, 9) | 2026-04-14 → 2026-06-11
TEST: (42, 22, 9) | 2026-06-12 → 2026-08-12


In [17]:
assert dates_train.iloc[-1] < dates_val.iloc[0]

assert dates_val.iloc[-1] < dates_test.iloc[0]

assert len(
    set(dates_train)
    & set(dates_val)
) == 0

assert len(
    set(dates_val)
    & set(dates_test)
) == 0

assert len(
    set(dates_train)
    & set(dates_test)
) == 0

print(
    "\nChronological split validation: PASSED"
)


Chronological split validation: PASSED


In [21]:
# ============================================================
# SCALE ALL SPLITS AND RESTORE 3D SHAPE
# ============================================================

# Transform using the scaler fitted on TRAIN only
X_train_scaled_2d = scaler.transform(X_train_2d)
X_val_scaled_2d = scaler.transform(X_val_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)

# Restore original TCN shape:
# (samples, timesteps, features)

X_train_scaled = X_train_scaled_2d.reshape(
    X_train.shape
)

X_val_scaled = X_val_scaled_2d.reshape(
    X_val.shape
)

X_test_scaled = X_test_scaled_2d.reshape(
    X_test.shape
)

print("=" * 60)
print("SCALED DATASET")
print("=" * 60)

print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

SCALED DATASET
Train: (124, 22, 9)
Validation: (42, 22, 9)
Test: (42, 22, 9)


In [22]:
X_train_scaled = X_train_scaled_2d.reshape(
    X_train.shape
)

X_val_scaled = X_val_scaled_2d.reshape(
    X_val.shape
)

X_test_scaled = X_test_scaled_2d.reshape(
    X_test.shape
)

print(
    "Scaled train:",
    X_train_scaled.shape
)

print(
    "Scaled validation:",
    X_val_scaled.shape
)

print(
    "Scaled test:",
    X_test_scaled.shape
)

Scaled train: (124, 22, 9)
Scaled validation: (42, 22, 9)
Scaled test: (42, 22, 9)


In [23]:
train_scaled_2d = X_train_scaled.reshape(
    -1,
    n_features
)

train_means = train_scaled_2d.mean(
    axis=0
)

train_stds = train_scaled_2d.std(
    axis=0
)

print("=" * 60)
print("TRAIN SCALING VALIDATION")
print("=" * 60)

for feature_idx in range(n_features):

    print(
        f"Feature {feature_idx + 1}: "
        f"mean={train_means[feature_idx]:.6f}, "
        f"std={train_stds[feature_idx]:.6f}"
    )

TRAIN SCALING VALIDATION
Feature 1: mean=0.000001, std=0.999998
Feature 2: mean=0.000000, std=0.999998
Feature 3: mean=-0.000000, std=0.999998
Feature 4: mean=-0.000000, std=1.000002
Feature 5: mean=-0.000000, std=0.999998
Feature 6: mean=0.000001, std=0.999998
Feature 7: mean=-0.000001, std=0.999999
Feature 8: mean=-0.000001, std=1.000002
Feature 9: mean=-0.000000, std=1.000000


In [24]:
print("=" * 60)
print("SCALER LEAKAGE CHECK")
print("=" * 60)

print(
    "Scaler was fitted using:",
    X_train_2d.shape
)

print(
    "Validation data used for fitting: NO"
)

print(
    "Test data used for fitting: NO"
)

print(
    "\nScaler leakage validation: PASSED"
)

SCALER LEAKAGE CHECK
Scaler was fitted using: (2728, 9)
Validation data used for fitting: NO
Test data used for fitting: NO

Scaler leakage validation: PASSED


In [25]:
assert X_train_scaled.shape == X_train.shape
assert X_val_scaled.shape == X_val.shape
assert X_test_scaled.shape == X_test.shape

assert np.isfinite(
    X_train_scaled
).all()

assert np.isfinite(
    X_val_scaled
).all()

assert np.isfinite(
    X_test_scaled
).all()

print("=" * 60)
print("STEP 3 SCALING VALIDATION")
print("=" * 60)

print(
    "Train shape:",
    X_train_scaled.shape
)

print(
    "Validation shape:",
    X_val_scaled.shape
)

print(
    "Test shape:",
    X_test_scaled.shape
)

print(
    "\nAll scaled values finite: PASSED"
)

print(
    "Training-only scaler fitting: PASSED"
)

print(
    "STEP 3: PASSED"
)

STEP 3 SCALING VALIDATION
Train shape: (124, 22, 9)
Validation shape: (42, 22, 9)
Test shape: (42, 22, 9)

All scaled values finite: PASSED
Training-only scaler fitting: PASSED
STEP 3: PASSED


In [26]:
print("=" * 60)
print("STEP 3 FINAL VALIDATION")
print("=" * 60)

assert X_train_scaled.shape == X_train.shape
assert X_val_scaled.shape == X_val.shape
assert X_test_scaled.shape == X_test.shape

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()
assert np.isfinite(X_test_scaled).all()

assert scaler.n_features_in_ == 9

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

print("\nAll scaled values finite: PASSED")
print("Scaler fitted on 9 features: PASSED")
print("Training-only scaling: PASSED")
print("\nSTEP 3: PASSED")

STEP 3 FINAL VALIDATION
Train shape: (124, 22, 9)
Validation shape: (42, 22, 9)
Test shape: (42, 22, 9)

All scaled values finite: PASSED
Scaler fitted on 9 features: PASSED
Training-only scaling: PASSED

STEP 3: PASSED


In [27]:
import joblib
import os

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../data/processed/X_train_scaled.npy", X_train_scaled)
np.save("../data/processed/X_val_scaled.npy", X_val_scaled)
np.save("../data/processed/X_test_scaled.npy", X_test_scaled)

np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_val.npy", y_val)
np.save("../data/processed/y_test.npy", y_test)

dates_train.to_csv(
    "../data/processed/dates_train.csv",
    index=False
)

dates_val.to_csv(
    "../data/processed/dates_val.csv",
    index=False
)

dates_test.to_csv(
    "../data/processed/dates_test.csv",
    index=False
)

joblib.dump(
    scaler,
    "../models/aapl_scaler.joblib"
)

print("Step 3 artifacts saved successfully.")

Step 3 artifacts saved successfully.
